# RetainSpot Customer Churn Prediction Models

## Imports

In [46]:
# Utility
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tqdm
import time
import joblib

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Training and Evaluation
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

## Data Loading

In [15]:
# Reading csv files into dataframes
X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train.csv').values.ravel()
y_test = pd.read_csv('/content/y_test.csv').values.ravel()

In [16]:
X_train.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_male,Partner_yes,Dependents_yes,PhoneService_yes,MultipleLines_no phone service,MultipleLines_yes,...,StreamingTV_no internet service,StreamingTV_yes,StreamingMovies_no internet service,StreamingMovies_yes,Contract_one year,Contract_two year,PaperlessBilling_yes,PaymentMethod_credit card (automatic),PaymentMethod_electronic check,PaymentMethod_mailed check
0,0,54,86.20,4524.05,0,1,1,1,0,1,...,0,1,0,1,0,0,1,0,1,0
1,0,22,75.00,1573.95,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,70,74.10,5222.30,0,1,1,1,0,0,...,0,0,0,1,1,0,0,1,0,0
3,0,10,95.25,1021.55,1,0,0,1,0,1,...,0,0,0,1,0,0,1,0,1,0
4,0,47,94.90,4615.25,0,0,0,1,0,1,...,0,1,0,0,1,0,0,1,0,0


In [17]:
# Data has already been preprocessed, just need to scale few numerical columns
scaler = StandardScaler()
X_train[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler.fit_transform(X_train[['tenure', 'MonthlyCharges', 'TotalCharges']])
X_test[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler.transform(X_test[['tenure', 'MonthlyCharges', 'TotalCharges']])

# Show X_train after scaling:
X_train.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_male,Partner_yes,Dependents_yes,PhoneService_yes,MultipleLines_no phone service,MultipleLines_yes,...,StreamingTV_no internet service,StreamingTV_yes,StreamingMovies_no internet service,StreamingMovies_yes,Contract_one year,Contract_two year,PaperlessBilling_yes,PaymentMethod_credit card (automatic),PaymentMethod_electronic check,PaymentMethod_mailed check
0,0,0.872250,0.702054,0.978287,0,1,1,1,0,1,...,0,1,0,1,0,0,1,0,1,0
1,0,-0.428418,0.330642,-0.316933,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1.522583,0.300797,1.284849,0,1,1,1,0,0,...,0,0,0,1,1,0,0,1,0,0
3,0,-0.916169,1.002169,-0.559461,1,0,0,1,0,1,...,0,0,0,1,0,0,1,0,1,0
4,0,0.587728,0.990562,1.018328,0,0,0,1,0,1,...,0,1,0,0,1,0,0,1,0,0


### Training Base Models

In [42]:
seed = 42

# BASE MODELS TO TRAIN
models = {
    'LogRegression': LogisticRegression(class_weight='balanced', random_state=seed),
    'LinearSVC': LinearSVC(class_weight='balanced', random_state=seed),
    'KNN': KNeighborsClassifier(),
    'DTree': DecisionTreeClassifier(class_weight='balanced', random_state=seed),
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=seed),
    'XGB': XGBClassifier(random_state=seed),
}

param_grids = {
    'LogRegression': {
        'model__C': [0.01, 0.1, 1, 10, 100],
        'model__penalty': ['l2'],
        'model__solver': ['lbfgs', 'liblinear'],
        'model__class_weight': [None, 'balanced']
    },
    'LinearSVC': {
        'model__C': [0.01, 0.1, 1, 10],
        'model__class_weight': [None, 'balanced'],
        'model__loss': ['hinge', 'squared_hinge']
    },
    'KNN': {
        'model__n_neighbors': [3, 5, 7, 9, 11],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['euclidean', 'manhattan']
    },
    'DTree': {
        'model__max_depth': [3, 5, 7, 10, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__class_weight': [None, 'balanced']
    },
    'RandomForest': {
        'model__n_estimators': [25, 50, 100, 200, 300],
        'model__max_depth': [5, 10, 15, None],
        'model__min_samples_split': [2, 5, 10],
        'model__class_weight': [None, 'balanced']
    },
    'XGB': {
        'model__n_estimators': [50, 100, 200, 300],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.01, 0.1, 0.3],
        'model__scale_pos_weight': [1, 3, 5]
    }
}

# DEFINE PIPELINE
def create_pipeline(model):
  pipeline = Pipeline([
      ('scaler', StandardScaler()),
      ('model', model)
  ])
  return pipeline

# MODEL TRAINING METHOD
def train_model(model, X_train, y_train, X_test, y_test, model_name):
  # Train the model
  start_time = time.time()
  model.fit(X_train, y_train)
  train_time = time.time() - start_time

  # Get model predictions
  y_pred = model.predict(X_test)

  # Get Model Metrics
  metrics = {
      'Model': model_name,
      'Accuracy': accuracy_score(y_test, y_pred),
      'Precision': precision_score(y_test, y_pred),
      'Recall': recall_score(y_test, y_pred),
      'F1': f1_score(y_test, y_pred),
      'ROC_AUC': roc_auc_score(y_test, y_pred),
      'Training Time': train_time
  }

  return metrics, y_pred

In [43]:
# TRAIN BASE MODELS
def train_base_models(models):
  results = {}
  for model_name, model in models.items():
    print(f'TRAINING BASE MODEL {model_name}')
    metrics, y_pred = train_model(model, X_train, y_train, X_test, y_test, model_name)
    print(f'TRAINING COMPLETE.')
    print(f'METRICS:')
    print(f'ACCURACY: {metrics["Accuracy"]:.3f}')
    print(f'PRECISION: {metrics["Precision"]:.3f}')
    print(f'RECALL: {metrics["Recall"]:.3f}')
    print(f'F1: {metrics["F1"]:.3f}')
    print(f'ROC AUC: {metrics["ROC_AUC"]:.3f}')
    print(f'Training Time: {metrics["Training Time"]:.3f} sec\n')

train_base_models(models)

TRAINING BASE MODEL LogRegression
TRAINING COMPLETE.
METRICS:
ACCURACY: 0.742
PRECISION: 0.509
RECALL: 0.818
F1: 0.628
ROC AUC: 0.766
Training Time: 0.261 sec

TRAINING BASE MODEL LinearSVC
TRAINING COMPLETE.
METRICS:
ACCURACY: 0.736
PRECISION: 0.502
RECALL: 0.824
F1: 0.623
ROC AUC: 0.764
Training Time: 0.094 sec

TRAINING BASE MODEL KNN
TRAINING COMPLETE.
METRICS:
ACCURACY: 0.763
PRECISION: 0.557
RECALL: 0.535
F1: 0.546
ROC AUC: 0.690
Training Time: 0.007 sec

TRAINING BASE MODEL DTree
TRAINING COMPLETE.
METRICS:
ACCURACY: 0.734
PRECISION: 0.500
RECALL: 0.484
F1: 0.492
ROC AUC: 0.654
Training Time: 0.084 sec

TRAINING BASE MODEL RandomForest
TRAINING COMPLETE.
METRICS:
ACCURACY: 0.780
PRECISION: 0.620
RECALL: 0.441
F1: 0.516
ROC AUC: 0.672
Training Time: 1.475 sec

TRAINING BASE MODEL XGB
TRAINING COMPLETE.
METRICS:
ACCURACY: 0.778
PRECISION: 0.605
RECALL: 0.476
F1: 0.533
ROC AUC: 0.682
Training Time: 0.535 sec



### Train Models with GridSearchCV

In [44]:
# TRAIN WITH GRID SEARCH
def train_with_gridsearch(models, X_train, y_train, X_test, y_test):

  results = []
  best_models = {}
  predictions = {}

  for model_name, model in models.items():

    # Create pipeline
    pipeline = create_pipeline(model)

    grid_search = GridSearchCV(
        pipeline,
        param_grids[model_name],
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=seed),
        scoring='f1',
        n_jobs=1,
        verbose=0
    )

    # Train using grid search
    print(f'TRAINING MODEL {model_name} WITH GRID SEARCH')
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    train_time = time.time() - start_time
    print(f'TRAINING COMPLETE.')

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    print(f'Best Parameters: {best_params}')
    print(f'Best CV Score: {grid_search.best_score_:.3f}')

    # Get predictions
    y_pred = best_model.predict(X_test)

    # Get Model Metrics
    metrics = {
      'Model': model_name,
      'Accuracy': accuracy_score(y_test, y_pred),
      'Precision': precision_score(y_test, y_pred),
      'Recall': recall_score(y_test, y_pred),
      'F1': f1_score(y_test, y_pred),
      'ROC_AUC': roc_auc_score(y_test, y_pred),
      'Training Time': train_time
    }

    print(f'METRICS:')
    print(f'ACCURACY: {metrics["Accuracy"]:.3f}')
    print(f'PRECISION: {metrics["Precision"]:.3f}')
    print(f'RECALL: {metrics["Recall"]:.3f}')
    print(f'F1: {metrics["F1"]:.3f}')
    print(f'ROC AUC: {metrics["ROC_AUC"]:.3f}')
    print(f'Training Time: {metrics["Training Time"]:.3f} sec\n')

    results.append(metrics)
    best_models[model_name] = best_model
    predictions[model_name] = y_pred

  return results, best_models, predictions


results = []
best_models = {}
predictions = {}

results, best_models, predictions = train_with_gridsearch(models, X_train, y_train, X_test, y_test)

TRAINING MODEL LogRegression WITH GRID SEARCH
TRAINING COMPLETE.
Best Parameters: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV Score: 0.628
METRICS:
ACCURACY: 0.742
PRECISION: 0.509
RECALL: 0.813
F1: 0.626
ROC AUC: 0.765
Training Time: 7.278 sec

TRAINING MODEL LinearSVC WITH GRID SEARCH


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

TRAINING COMPLETE.
Best Parameters: {'model__C': 0.01, 'model__class_weight': 'balanced', 'model__loss': 'squared_hinge'}
Best CV Score: 0.625
METRICS:
ACCURACY: 0.736
PRECISION: 0.502
RECALL: 0.824
F1: 0.623
ROC AUC: 0.764
Training Time: 7.570 sec

TRAINING MODEL KNN WITH GRID SEARCH
TRAINING COMPLETE.
Best Parameters: {'model__metric': 'manhattan', 'model__n_neighbors': 11, 'model__weights': 'uniform'}
Best CV Score: 0.577
METRICS:
ACCURACY: 0.787
PRECISION: 0.609
RECALL: 0.551
F1: 0.579
ROC AUC: 0.712
Training Time: 12.881 sec

TRAINING MODEL DTree WITH GRID SEARCH
TRAINING COMPLETE.
Best Parameters: {'model__class_weight': 'balanced', 'model__max_depth': 7, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Best CV Score: 0.608
METRICS:
ACCURACY: 0.735
PRECISION: 0.501
RECALL: 0.797
F1: 0.615
ROC AUC: 0.755
Training Time: 15.717 sec

TRAINING MODEL RandomForest WITH GRID SEARCH
TRAINING COMPLETE.
Best Parameters: {'model__class_weight': 'balanced', 'model__max_depth': 10,

#### Save the Random Forest Model

In [47]:

model = best_models["RandomForest"]
joblib.dump(model, "churn_model.pkl")

['churn_model.pkl']

#### Adding Churn Scores (Model Probabilities) to Original Dataset

In [71]:
# Get original dataset and process
target = "Churn"
data = pd.read_csv('/content/telco_noisy_feedback_prep.csv')
data_clean = data.drop("Churn", axis=1)
data_clean = data_clean.drop(columns=[
    "CustomerFeedback",
    "sentiment",
    "feedback_length",
    "HasFeedback",
    "Unnamed: 0",
    "Unnamed: 0.1",
    "Churn",
    "customerID",
], errors="ignore")
data_clean.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,female,0,yes,no,1,no,no phone service,dsl,no,yes,no,no,no,no,month-to-month,yes,electronic check,29.85,29.85
1,male,0,no,no,34,yes,no,dsl,yes,no,yes,no,no,no,one year,no,mailed check,56.95,1889.50
2,male,0,no,no,2,yes,no,dsl,yes,yes,no,no,no,no,month-to-month,yes,mailed check,53.85,108.15
3,male,0,no,no,45,no,no phone service,dsl,yes,no,yes,yes,no,no,one year,no,bank transfer (automatic),42.30,1840.75
4,female,0,no,no,2,yes,no,fiber optic,no,no,no,no,no,no,month-to-month,yes,electronic check,70.70,151.65


In [69]:
print("Numeric columns:")
print(data_clean.select_dtypes(include=["int64", "float64"]).columns.tolist())

print("\nCategorical columns:")
print(data_clean.select_dtypes(include=["object"]).columns.tolist())

Numeric columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges']

Categorical columns:
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']


In [73]:
data_encoded = pd.get_dummies(data_clean, drop_first=True)
data_encoded = data_encoded.astype(int)
data_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_male,Partner_yes,Dependents_yes,PhoneService_yes,MultipleLines_no phone service,MultipleLines_yes,...,StreamingTV_no internet service,StreamingTV_yes,StreamingMovies_no internet service,StreamingMovies_yes,Contract_one year,Contract_two year,PaperlessBilling_yes,PaymentMethod_credit card (automatic),PaymentMethod_electronic check,PaymentMethod_mailed check
0,0,1,29,29,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56,1889,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53,108,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42,1840,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70,151,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


In [74]:
bool_cols = data_encoded.select_dtypes(include="bool").columns
data_encoded[bool_cols] = data_encoded[bool_cols].astype(int)

data_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_male,Partner_yes,Dependents_yes,PhoneService_yes,MultipleLines_no phone service,MultipleLines_yes,...,StreamingTV_no internet service,StreamingTV_yes,StreamingMovies_no internet service,StreamingMovies_yes,Contract_one year,Contract_two year,PaperlessBilling_yes,PaymentMethod_credit card (automatic),PaymentMethod_electronic check,PaymentMethod_mailed check
0,0,1,29,29,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56,1889,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53,108,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42,1840,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70,151,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


In [77]:
scaler = StandardScaler()
data_encoded[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler.fit_transform(data_encoded[['tenure', 'MonthlyCharges', 'TotalCharges']])
data_encoded.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_male,Partner_yes,Dependents_yes,PhoneService_yes,MultipleLines_no phone service,MultipleLines_yes,...,StreamingTV_no internet service,StreamingTV_yes,StreamingMovies_no internet service,StreamingMovies_yes,Contract_one year,Contract_two year,PaperlessBilling_yes,PaymentMethod_credit card (automatic),PaymentMethod_electronic check,PaymentMethod_mailed check
0,0,-1.280248,-1.174362,-0.994363,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,0.064303,-0.276951,-0.173753,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,-1.239504,-0.376663,-0.959509,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,0.512486,-0.742275,-0.195372,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,-1.239504,0.188374,-0.940538,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


##### Get Model Probabilities

In [79]:
y_proba = model.predict_proba(data_encoded)

array([[0.23522507, 0.76477493],
       [0.94181388, 0.05818612],
       [0.33415453, 0.66584547],
       [0.912169  , 0.087831  ],
       [0.17768248, 0.82231752],
       [0.09117142, 0.90882858],
       [0.40493871, 0.59506129],
       [0.67228959, 0.32771041],
       [0.26988689, 0.73011311],
       [0.94481955, 0.05518045],
       [0.72890423, 0.27109577],
       [0.95006649, 0.04993351],
       [0.6534609 , 0.3465391 ],
       [0.35685985, 0.64314015],
       [0.42374737, 0.57625263]])

In [83]:
# Append churn scores to dataset
data["Churn Score"] = y_proba[:, 1] * 100
data = data.drop("Churn", axis=1)
data.head()

,Unnamed: 0.1,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,CustomerFeedback,feedback_length,sentiment,HasFeedback,Churn Score
0,0,0,7590-vhveg,female,0,yes,no,1,no,no phone service,...,month-to-month,yes,electronic check,29.85,29.85,NaN,0,0.0,False,76.477493
1,1,1,5575-gnvde,male,0,no,no,34,yes,no,...,one year,no,mailed check,56.95,1889.50,NaN,0,0.0,False,5.818612
2,2,2,3668-qpybk,male,0,no,no,2,yes,no,...,month-to-month,yes,mailed check,53.85,108.15,NaN,0,0.0,False,66.584547
3,3,3,7795-cfocw,male,0,no,no,45,no,no phone service,...,one year,no,bank transfer (automatic),42.30,1840.75,NaN,0,0.0,False,8.783100
4,4,4,9237-hqitu,female,0,no,no,2,yes,no,...,month-to-month,yes,electronic check,70.70,151.65,NaN,0,0.0,False,82.231752


In [84]:
data.to_csv("data_with_churn_score.csv", index=False)